# NBA FG3M · Análisis Exploratorio de Datos

Notebook dedicado exclusivamente al **análisis exploratorio de datos** para `TARGET_FG3M`, los triples convertidos por un jugador en el partido objetivo.

### Diseño de esta versión

- Gráficos estáticos con `matplotlib`, compatibles con el renderizado de GitHub.
- Tablas estándar de `pandas`, sin JavaScript ni widgets.
- Capa de interpretación enriquecida con la especificación funcional del **Dataset Master NBA, versión 1.0, 22 de junio de 2026**.
- Normalización Min-Max al intervalo `[0,1]`.
- Calidad, distribución, asociaciones, dispersión con tendencia y clustering jerárquico.
- Sin entrenamiento de modelos y sin exportación de artefactos.

> Para que GitHub muestre los gráficos, ejecuta todas las celdas antes de guardar y subir el `.ipynb`.


## 1. Marco funcional del dataset

La especificación define una observación como **un jugador en un partido objetivo**. Las variables explicativas deben utilizar únicamente información anterior a `TARGET_GAME_DATE`; los campos `TARGET_*` representan resultados observados en el partido objetivo.

| Elemento | Regla funcional |
|---|---|
| Unidad de observación | Jugador-partido |
| Llave lógica | `PLAYER_ID + TARGET_GAME_ID` |
| Bloque jugador | 419 campos |
| Bloque rival incorporado | 321 campos con prefijo `OPP_` |
| Esquema master esperado | 740 campos |
| Ventanas jugador | Temporada, L10, L20, último partido y tendencias L3/L5/L10 |
| Ventanas rival | Temporada, L3, L5 y L10 |
| Regla temporal | Toda feature debe ser estrictamente anterior al partido target |
| Target del EDA | `TARGET_FG3M` |
| Variables prohibidas como X | Otros `TARGET_*`, todos los `NEXT_GAME_*` y resultados del partido target |


## 2. Configuración visual y librerías

Tema claro inspirado en la NBA. Todas las figuras se generan como imágenes estáticas.


In [18]:
# @title
import warnings
warnings.filterwarnings("ignore")

import re
import json
import base64
import urllib.request
import urllib.error
from io import BytesIO
from getpass import getpass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import squareform
from IPython.display import display, Markdown

import plotly.graph_objects as go
import plotly.express as px

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 120)
pd.set_option("display.float_format", "{:,.3f}".format)

NAVY = "#0B1F33"
BLUE = "#2D6CDF"
RED = "#C8102E"
GOLD = "#F2B134"
TEAL = "#1F9D8A"
PURPLE = "#7158B8"
SLATE = "#5B677A"
LIGHT_BLUE = "#8EB8E5"
GRID = "#D6DEE8"
TEXT = "#17212B"
MUTED = "#667085"
WHITE = "#FFFFFF"

PALETTE = [BLUE, RED, GOLD, TEAL, PURPLE, SLATE, LIGHT_BLUE, "#D97706"]
CMAP_SEQ = LinearSegmentedColormap.from_list("nba_seq", ["#F7FAFC", LIGHT_BLUE, BLUE, NAVY])
CMAP_DIV = LinearSegmentedColormap.from_list("nba_div", [RED, "#F8FAFC", BLUE])

# Matplotlib rcParams (kept for potential future use or if other parts still use it)
plt.rcParams.update({
    "figure.facecolor": WHITE,
    "axes.facecolor": WHITE,
    "savefig.facecolor": WHITE,
    "axes.edgecolor": GRID,
    "axes.labelcolor": TEXT,
    "axes.titlecolor": NAVY,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "axes.titlepad": 12,
    "axes.labelsize": 10,
    "axes.grid": True,
    "grid.color": GRID,
    "grid.linestyle": "--",
    "grid.linewidth": 0.7,
    "grid.alpha": 0.65,
    "xtick.color": MUTED,
    "ytick.color": MUTED,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.frameon": False,
    "legend.fontsize": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.dpi": 120,
    "figure.figsize": (11, 5.5),
    "font.family": "DejaVu Sans",
})

def section(title, subtitle=""):
    text = f"### {title}"
    if subtitle:
        text += f"\n\n{subtitle}"
    display(Markdown(text))

def note(message, label="Nota"):
    display(Markdown(f"> **{label}:** {message}"))

def metric_table(values):
    display(pd.DataFrame({"Métrica": list(values.keys()), "Valor": list(values.values())}))

def finish_axes(ax, xlabel=None, ylabel=None):
    ax.grid(axis="y", alpha=0.65)
    ax.grid(axis="x", alpha=0.25)
    if xlabel:
        ax.set_xlabel(xlabel)
    if ylabel:
        ax.set_ylabel(ylabel)
    return ax

def finish_plotly_layout(fig, title=None, xlabel=None, ylabel=None):
    fig.update_layout(
        title_text=title,
        title_x=0.5,
        plot_bgcolor=WHITE,
        paper_bgcolor=WHITE,
        font=dict(color=TEXT),
        xaxis_title=xlabel,
        yaxis_title=ylabel,
        xaxis_gridcolor=GRID,
        yaxis_gridcolor=GRID,
        xaxis_showgrid=True,
        yaxis_showgrid=True,
        margin=dict(l=40, r=40, t=80, b=40) # Adjust margins as needed
    )
    fig.update_xaxes(zeroline=False, showline=False)
    fig.update_yaxes(zeroline=False, showline=False)
    return fig

def show_df(frame, rows=20):
    display(frame.head(rows))

print("Configuración visual cargada. Salidas estáticas compatibles con GitHub.")

Configuración visual cargada. Salidas estáticas compatibles con GitHub.


## 3. Configuración operativa


In [2]:
# @title
CONFIG = {
    "GITHUB_OWNER": "SebaSRomanH",
    "GITHUB_REPO": "NBA",
    "GITHUB_BRANCH": "main",
    "GITHUB_PATH": "Data/dataset_nba.xlsx",
    "SHEET_NAME": "Hoja 1",
    "DECIMAL": ",",

    "TARGET_COL": "TARGET_FG3M",

    "SPEC_TITLE": "Dataset Master NBA - Especificación funcional, lógica temporal y diccionario de variables",
    "SPEC_VERSION": "1.0",
    "SPEC_DATE": "2026-06-22",
    "SPEC_STATUS": "Diseño previo a implementación",

    "CLUSTER_N_GROUPS": 6,
    "CLUSTER_LINKAGE": "ward",
    "CLUSTER_MAX_ROWS": 5000,

    "SCALING_MIN": 0.0,
    "SCALING_MAX": 1.0,

    "SCATTER_VARIABLE": "AVG_FG3A_L20",
    "SCATTER_MAX_POINTS": 5000,
    "SCATTER_GALLERY_TOP": 4,

    "NULL_THRESHOLD": 0.30,
    "SEED": 42,

    "ID_COLS": [
        "PLAYER_ID", "TARGET_GAME_ID", "LAST_GAME_ID",
        "TARGET_TEAM_ID", "OPP_TEAM_ID", "PLAYER_NAME_RAW"
    ],
}

np.random.seed(CONFIG["SEED"])

metric_table({
    "Variable dependiente": CONFIG["TARGET_COL"],
    "Número de clusters": CONFIG["CLUSTER_N_GROUPS"],
    "Método de enlace": CONFIG["CLUSTER_LINKAGE"],
    "Escala": "[0,1]",
    "Especificación": f"v{CONFIG['SPEC_VERSION']} - {CONFIG['SPEC_DATE']}",
})

note(
    "El notebook conserva `df` en escala original, crea `df_scaled` con variables numéricas en `[0,1]` "
    "y utiliza `df_eda` para mantener `TARGET_FG3M` en unidades reales.",
    "Decisión metodológica"
)


,Métrica,Valor
0,Variable dependiente,TARGET_FG3M
1,Número de clusters,6
2,Método de enlace,ward
3,Escala,"[0,1]"
4,Especificación,v1.0 - 2026-06-22


> **Decisión metodológica:** El notebook conserva `df` en escala original, crea `df_scaled` con variables numéricas en `[0,1]` y utiliza `df_eda` para mantener `TARGET_FG3M` en unidades reales.

## 4. Autenticación y carga desde GitHub

El token se solicita de forma segura y no se escribe en el notebook. En Google Colab puede guardarse como secreto `GITHUB_PAT`.


In [3]:
# @title
def get_github_token():
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_PAT")
        if token:
            print("Token cargado desde Colab Secrets.")
            return token
    except Exception:
        pass

    token = getpass("GitHub Personal Access Token: ")
    if not token:
        raise ValueError("No se proporcionó un token de GitHub.")
    return token

def fetch_github_file(owner, repo, path, branch, token):
    api_url = f"https://api.github.com/repos/{owner}/{repo}/contents/{path}?ref={branch}"
    request = urllib.request.Request(
        api_url,
        headers={
            "Authorization": f"Bearer {token}",
            "Accept": "application/vnd.github+json",
            "X-GitHub-Api-Version": "2022-11-28",
        },
    )

    with urllib.request.urlopen(request, timeout=30) as response:
        metadata = json.loads(response.read())

    if metadata.get("encoding") == "base64" and metadata.get("content"):
        return base64.b64decode(metadata["content"].replace("\n", ""))

    if metadata.get("download_url"):
        request_raw = urllib.request.Request(
            metadata["download_url"],
            headers={"Authorization": f"Bearer {token}"}
        )
        with urllib.request.urlopen(request_raw, timeout=60) as response:
            return response.read()

    raise RuntimeError("GitHub no devolvió el contenido del archivo.")

def normalize_column_names(data):
    mapping = {col: str(col).strip().upper().replace(" ", "_") for col in data.columns}
    return data.rename(columns=mapping)

def parse_excel(raw_bytes, sheet_name, decimal=","):
    data = pd.read_excel(BytesIO(raw_bytes), sheet_name=sheet_name, engine="openpyxl")

    for col in data.select_dtypes(include="object").columns:
        cleaned = (
            data[col].astype(str).str.strip().str.replace(decimal, ".", regex=False)
        )
        converted = pd.to_numeric(cleaned, errors="coerce")
        valid_original = data[col].notna().sum()
        valid_converted = converted.notna().sum()

        if valid_original > 0 and valid_converted / valid_original >= 0.90:
            data[col] = converted

    return normalize_column_names(data)

GITHUB_TOKEN = get_github_token()

raw_file = fetch_github_file(
    CONFIG["GITHUB_OWNER"],
    CONFIG["GITHUB_REPO"],
    CONFIG["GITHUB_PATH"],
    CONFIG["GITHUB_BRANCH"],
    GITHUB_TOKEN,
)

df_original = parse_excel(raw_file, CONFIG["SHEET_NAME"], CONFIG["DECIMAL"])
df = df_original.copy()

section("Dataset cargado", f"{CONFIG['GITHUB_OWNER']}/{CONFIG['GITHUB_REPO']} - {CONFIG['GITHUB_PATH']}")
metric_table({
    "Filas": f"{df.shape[0]:,}",
    "Columnas": f"{df.shape[1]:,}",
    "Memoria": f"{df.memory_usage(deep=True).sum() / 1e6:,.1f} MB",
})
show_df(df, 5)


GitHub Personal Access Token: ··········


### Dataset cargado

SebaSRomanH/NBA - Data/dataset_nba.xlsx

,Métrica,Valor
0,Filas,"3,122"
1,Columnas,419
2,Memoria,13.6 MB


,PLAYER_ID,PLAYER_NAME,PLAYER_NAME_RAW,TARGET_GAME_ID,TARGET_GAME_DATE,TARGET_SEASON_YEAR,TARGET_SEASON_TYPE,TARGET_TEAM_ID,TARGET_TEAM_ABBREVIATION,TARGET_TEAM_NAME,TARGET_MATCHUP,TARGET_HOME_AWAY,TARGET_OPP_TEAM_ABBREVIATION,TEAM_ABBREVIATION,GAME_DATE,NEXT_GAME_DATE,NEXT_OPP_TEAM_ABBREVIATION,BIRTH_DATE,AGE,HEIGHT,WEIGHT,PLAYER_POSITION,SCHOOL,GAMES_PLAYED,GAMES_PLAYED_L10,GAMES_PLAYED_L20,LAST_GAME_ID,LAST_GAME_DATE,DAYS_SINCE_LAST_GAME,LAST_GAME_TEAM_ABBREVIATION,LAST_GAME_OPP_TEAM_ABBREVIATION,LAST_GAME_MIN,LAST_GAME_PTS,LAST_GAME_AST,LAST_GAME_REB,LAST_GAME_FG3M,LAST_GAME_STL,LAST_GAME_BLK,LAST_GAME_TOV,MIN_TOTAL,...,FG3M_TREND_L3,FG3_PCT_TREND_L3,PTS_TREND_L5,AST_TREND_L5,REB_TREND_L5,STL_TREND_L5,BLK_TREND_L5,TOV_TREND_L5,FG3M_TREND_L5,FG3_PCT_TREND_L5,PTS_TREND_L10,AST_TREND_L10,REB_TREND_L10,STL_TREND_L10,BLK_TREND_L10,TOV_TREND_L10,FG3M_TREND_L10,FG3_PCT_TREND_L10,TARGET_MIN,TARGET_PTS,TARGET_AST,TARGET_REB,TARGET_FG3M,TARGET_FGA,TARGET_FGM,TARGET_FG3A,TARGET_STL,TARGET_BLK,TARGET_TOV,NEXT_GAME_MIN,NEXT_GAME_PTS,NEXT_GAME_AST,NEXT_GAME_REB,NEXT_GAME_FG3M,NEXT_GAME_FGA,NEXT_GAME_FGM,NEXT_GAME_FG3A,NEXT_GAME_STL,NEXT_GAME_BLK,NEXT_GAME_TOV
0,1630543,Isaiah Jackson,Isaiah Jackson,22500717,2026-02-03,2025-26,REGULAR,1610612754,IND,Indiana Pacers,IND vs. UTA,HOME,UTA,IND,2026-02-02,2026-02-03,UTA,2002-01-10,24,6-8,205.000,F,Kentucky,37,10,20,22500713,2026-02-02,1,IND,HOU,24.853,6,4,6,0,4,2,2,614.890,...,0.000,0.000,0.800,0.900,0.000,0.800,0.400,0.100,0.000,0.000,-0.121,0.200,0.212,0.218,0.073,-0.006,0.000,0.000,23.183,11,1,10,0,6,4,0,1,1,1,23.183,11,1,10,0,6,4,0,1,1,1
1,1630643,Jay Huff,Jay Huff,22500717,2026-02-03,2025-26,REGULAR,1610612754,IND,Indiana Pacers,IND vs. UTA,HOME,UTA,IND,2026-02-02,2026-02-03,UTA,1997-08-25,28,7-1,240.000,C,Virginia,50,10,20,22500713,2026-02-02,1,IND,HOU,11.713,6,1,3,2,0,2,0,968.948,...,0.500,0.067,2.500,0.100,1.100,0.000,0.600,0.200,0.600,0.004,-0.533,0.006,-0.109,-0.006,-0.085,0.024,0.091,0.018,24.817,12,3,6,2,9,5,4,0,0,1,24.817,12,3,6,2,9,5,4,0,0,1
2,1641716,Jarace Walker,Jarace Walker,22500717,2026-02-03,2025-26,REGULAR,1610612754,IND,Indiana Pacers,IND vs. UTA,HOME,UTA,IND,2026-02-02,2026-02-03,UTA,2003-09-04,22,6-7,235.000,F,Houston,50,10,20,22500713,2026-02-02,1,IND,HOU,22.583,12,1,4,2,1,0,0,"1,174.523",...,0.000,-0.083,-2.900,-0.300,0.300,-0.700,-0.100,-0.600,-0.300,-0.017,0.473,-0.048,0.436,0.115,-0.061,-0.030,-0.061,0.009,31.968,24,4,6,1,17,8,7,1,0,8,31.968,24,4,6,1,17,8,7,1,0,8
3,1641767,Ben Sheppard,Ben Sheppard,22500717,2026-02-03,2025-26,REGULAR,1610612754,IND,Indiana Pacers,IND vs. UTA,HOME,UTA,IND,2026-02-02,2026-02-03,UTA,2001-07-16,24,6-6,190.000,G,Belmont,39,10,20,22500713,2026-02-02,1,IND,HOU,7.317,2,2,0,0,0,0,0,831.732,...,-0.500,-0.500,-0.200,0.300,0.500,0.100,0.000,0.100,-0.200,-0.083,-0.067,0.055,0.133,-0.006,-0.030,0.048,0.024,0.022,28.617,8,1,1,1,7,3,4,0,0,2,28.617,8,1,1,1,7,3,4,0,0,2
4,1642277,Johnny Furphy,Johnny Furphy,22500717,2026-02-03,2025-26,REGULAR,1610612754,IND,Indiana Pacers,IND vs. UTA,HOME,UTA,IND,2026-02-02,2026-02-03,UTA,2004-12-08,21,6-8,200.000,G,Kansas,32,10,20,22500713,2026-02-02,1,IND,HOU,13.850,2,1,4,0,0,0,0,567.350,...,0.000,0.000,-1.000,-0.400,-1.500,-0.400,-0.200,0.300,0.000,0.000,-0.806,-0.030,-0.303,-0.030,-0.145,-0.097,-0.212,-0.074,28.882,14,6,7,2,11,6,7,0,0,2,28.882,14,6,7,2,11,6,7,0,0,2


## 5. Metadata funcional derivada de la especificación PDF

Esta capa incorpora directamente las reglas del documento para que el notebook sea autocontenido.

### Principios implementados

- `TARGET_FG3M` es el resultado de triples convertidos en el partido objetivo.
- Las features de jugador sin sufijo de ventana representan temporada hasta antes del target.
- `_L10` y `_L20` representan los últimos 10 y 20 partidos previos del jugador.
- `LAST_GAME_*` representa el partido inmediatamente anterior.
- `*_TREND_L3/L5/L10` representa una pendiente lineal; positivo indica crecimiento hacia el presente.
- `OPP_FOR_*` describe producción del rival.
- `OPP_AGAINST_*` describe estadísticas registradas por equipos que enfrentaron al rival.
- `NEXT_GAME_*` son aliases legacy de targets y deben excluirse de predictores.


In [4]:
# @title
DOCUMENT_SPEC = pd.DataFrame([
    ["Documento", CONFIG["SPEC_TITLE"]],
    ["Versión", CONFIG["SPEC_VERSION"]],
    ["Fecha", CONFIG["SPEC_DATE"]],
    ["Estado", CONFIG["SPEC_STATUS"]],
    ["Unidad de observación", "Jugador en un partido objetivo"],
    ["Llave lógica", "PLAYER_ID + TARGET_GAME_ID"],
    ["Campos jugador", 419],
    ["Campos rival incorporados", 321],
    ["Campos master esperados", 740],
    ["Regla temporal", "Toda feature usa información anterior a TARGET_GAME_DATE"],
    ["Target del EDA", CONFIG["TARGET_COL"]],
], columns=["Elemento", "Definición"])

display(DOCUMENT_SPEC)

NBA_METRICS = {
    "PLUS_MINUS": "diferencial plus/minus",
    "FG3_PCT": "porcentaje de triples",
    "FG2_PCT": "porcentaje de tiros de dos puntos",
    "FG_PCT": "porcentaje de tiros de campo",
    "FT_PCT": "porcentaje de tiros libres",
    "EFG_PCT": "porcentaje efectivo de tiro",
    "TS_PCT": "porcentaje de tiro verdadero",
    "FG3M": "triples convertidos",
    "FG3A": "triples intentados",
    "FGM": "tiros de campo convertidos",
    "FGA": "tiros de campo intentados",
    "FTM": "tiros libres convertidos",
    "FTA": "tiros libres intentados",
    "OREB": "rebotes ofensivos",
    "DREB": "rebotes defensivos",
    "REB": "rebotes totales",
    "AST": "asistencias",
    "TOV": "pérdidas de balón",
    "STL": "robos",
    "BLKA": "intentos de tiro bloqueados",
    "BLK": "bloqueos realizados",
    "PFD": "faltas personales recibidas o provocadas",
    "PF": "faltas personales cometidas",
    "PTS": "puntos",
    "MIN": "minutos jugados",
}

STATISTICS = {
    "AVG": "promedio",
    "MAX": "máximo",
    "MIN": "mínimo",
    "STDEV": "desviación estándar",
    "TOTAL": "suma",
    "PER_MIN": "tasa por minuto",
    "TREND": "tendencia lineal",
    "LAST_GAME": "valor del último partido",
}

CONTEXT_EXACT = {
    "PLAYER_ID": ("Clave", "Identidad", "Identificador NBA estable del jugador."),
    "PLAYER_NAME": ("Contexto", "Identidad", "Nombre canónico del jugador."),
    "PLAYER_NAME_RAW": ("Contexto", "Identidad", "Nombre original conservado para auditoría."),
    "TARGET_GAME_ID": ("Clave", "Partido objetivo", "Identificador del partido objetivo."),
    "TARGET_GAME_DATE": ("Clave temporal", "Partido objetivo", "Fecha del partido objetivo."),
    "TARGET_SEASON_YEAR": ("Contexto", "Partido objetivo", "Temporada NBA del partido objetivo."),
    "TARGET_SEASON_TYPE": ("Contexto", "Partido objetivo", "Tipo de temporada del partido objetivo."),
    "TARGET_HOME_AWAY": ("Predictor/contexto", "Partido objetivo", "Condición local o visitante."),
    "TARGET_OPP_TEAM_ABBREVIATION": ("Contexto/join", "Partido objetivo", "Abreviatura del rival."),
    "GAMES_PLAYED": ("Control/predictor", "Cobertura", "Partidos previos del jugador en la temporada."),
    "GAMES_PLAYED_L10": ("Control/predictor", "Cobertura", "Observaciones disponibles en la ventana de hasta 10 partidos."),
    "GAMES_PLAYED_L20": ("Control/predictor", "Cobertura", "Observaciones disponibles en la ventana de hasta 20 partidos."),
    "DAYS_SINCE_LAST_GAME": ("Predictor", "Último partido", "Días desde el último partido del jugador."),
}

TARGET_METRICS = {
    "TARGET_MIN", "TARGET_PTS", "TARGET_AST", "TARGET_REB", "TARGET_FG3M",
    "TARGET_FGA", "TARGET_FGM", "TARGET_FG3A", "TARGET_STL", "TARGET_BLK",
    "TARGET_TOV"
}

def detect_window(col):
    if col.startswith("LAST_GAME_"):
        return "Último partido"
    for window in ["L3", "L5", "L10", "L20"]:
        if re.search(fr"(?:^|_){window}(?:$|_)", col):
            return window
    if "_SEASON" in col:
        return "SEASON"
    if col.startswith("OPP_"):
        return "Contexto rival"
    if any(token in col for token in NBA_METRICS):
        return "SEASON"
    return "No aplica"

def detect_metric(col):
    for token in sorted(NBA_METRICS, key=len, reverse=True):
        if re.search(fr"(?:^|_){re.escape(token)}(?:$|_)", col):
            return token, NBA_METRICS[token]
    return None, "variable de contexto"

def detect_statistic(col):
    if col.startswith("LAST_GAME_"):
        return "LAST_GAME", STATISTICS["LAST_GAME"]
    if "_TREND_" in col:
        return "TREND", STATISTICS["TREND"]
    if "_PER_MIN" in col:
        return "PER_MIN", STATISTICS["PER_MIN"]
    for token in ["STDEV", "AVG", "MAX", "MIN"]:
        if col.startswith(token + "_") or f"_{token}_" in col:
            return token, STATISTICS[token]
    if col.endswith("_TOTAL") or "_TOTAL_" in col:
        return "TOTAL", STATISTICS["TOTAL"]
    return None, "valor"

def classify_role_family(col):
    if col in CONTEXT_EXACT:
        role, family, _ = CONTEXT_EXACT[col]
        return role, family
    if col.startswith("NEXT_GAME_"):
        return "Alias target", "Compatibilidad"
    if col in TARGET_METRICS:
        return "Target", "Resultado target"
    if col.startswith("TARGET_"):
        return "Contexto", "Partido objetivo"
    if col.startswith("OPP_TEAM_HAS_FULL_"):
        return "Control de calidad", "Cobertura rival"
    if col.startswith("OPP_TEAM_GAMES_PLAYED_"):
        return "Control/predictor", "Cobertura rival"
    if col.startswith("OPP_TEAM_"):
        return "Predictor/contexto", "Rival"
    if col.startswith("OPP_FOR_"):
        return "Predictor", "Producción rival"
    if col.startswith("OPP_AGAINST_"):
        return "Predictor", "Rendimiento contra el rival"
    if col.startswith("LAST_GAME_"):
        return "Predictor", "Último partido"
    if "_TREND_" in col:
        return "Predictor", "Tendencia jugador"
    if "_PER_MIN" in col:
        return "Predictor", "Tasa jugador"
    if col.endswith("_L10"):
        return "Predictor", "Agregado jugador L10"
    if col.endswith("_L20"):
        return "Predictor", "Agregado jugador L20"
    if col in CONFIG["ID_COLS"] or col.endswith("_ID"):
        return "Clave/contexto", "Identidad"
    return "Predictor/contexto", "Agregado jugador SEASON"

def build_description(col):
    if col in CONTEXT_EXACT:
        return CONTEXT_EXACT[col][2]

    metric_token, metric_name = detect_metric(col)

    if col.startswith("NEXT_GAME_"):
        return f"Alias legacy del resultado target de {metric_name}; excluir de variables explicativas."
    if col in TARGET_METRICS:
        return f"Valor observado de {metric_name} en el partido objetivo."

    window = detect_window(col)
    stat_token, stat_name = detect_statistic(col)

    if col.startswith("OPP_FOR_"):
        subject = "del equipo rival"
    elif col.startswith("OPP_AGAINST_"):
        subject = "registrado por los equipos que enfrentaron al rival"
    elif col.startswith("OPP_TEAM_"):
        subject = "del equipo rival"
    else:
        subject = "del jugador"

    if stat_token == "TREND":
        base = f"Tendencia lineal de {metric_name} {subject}"
    elif stat_token == "PER_MIN":
        base = f"{metric_name.capitalize()} por minuto {subject}"
    elif stat_token == "LAST_GAME":
        base = f"{metric_name.capitalize()} {subject} en su último partido previo"
    elif stat_token:
        base = f"{stat_name.capitalize()} de {metric_name} {subject}"
    else:
        base = f"{metric_name.capitalize()} {subject}"

    window_text = {
        "L3": "en los últimos 3 partidos previos",
        "L5": "en los últimos 5 partidos previos",
        "L10": "en los últimos 10 partidos previos",
        "L20": "en los últimos 20 partidos previos",
        "SEASON": "en la temporada hasta antes del partido objetivo",
        "Último partido": "en el último partido previo",
        "Contexto rival": "antes del partido objetivo",
        "No aplica": "",
    }.get(window, "")

    return (base + (" " + window_text if window_text else "") + ".").replace("..", ".")

def predictor_rule(col, role):
    if col.startswith("NEXT_GAME_"):
        return False, "Fuga de información: alias legacy del target."
    if col.startswith("TARGET_") and col != CONFIG["TARGET_COL"]:
        return False, "Resultado o contexto del partido objetivo; revisar disponibilidad."
    if col == CONFIG["TARGET_COL"]:
        return False, "Variable dependiente."
    if role.startswith("Clave"):
        return False, "Identificador; conservar solo para trazabilidad."
    return True, ""

metadata_rows = []
for col in df.columns:
    role, family = classify_role_family(col)
    _, metric_name = detect_metric(col)
    _, stat_name = detect_statistic(col)
    allowed, warning = predictor_rule(col, role)

    metadata_rows.append({
        "columna": col,
        "tipo_dato": str(df[col].dtype),
        "rol_pdf": role,
        "familia_pdf": family,
        "metrica_nba": metric_name,
        "estadistico": stat_name,
        "ventana_pdf": detect_window(col),
        "descripcion_breve": build_description(col),
        "predictor_permitido": allowed,
        "alerta": warning,
        "fuente_metadata": f"{CONFIG['SPEC_TITLE']} v{CONFIG['SPEC_VERSION']}",
    })

variable_dictionary = pd.DataFrame(metadata_rows)

section("Diccionario interpretado", "Metadata funcional aplicada a las columnas del dataset")
metric_table({
    "Columnas interpretadas": f"{len(variable_dictionary):,}",
    "Predictores permitidos": f"{variable_dictionary['predictor_permitido'].sum():,}",
    "Campos excluidos": f"{(~variable_dictionary['predictor_permitido']).sum():,}",
})

show_df(
    variable_dictionary[
        [
            "columna", "rol_pdf", "familia_pdf", "metrica_nba",
            "estadistico", "ventana_pdf", "descripcion_breve",
            "predictor_permitido", "alerta"
        ]
    ],
    30
)

example_name = "MIN_DREB_L10"
if example_name in variable_dictionary["columna"].values:
    display(variable_dictionary.loc[
        variable_dictionary["columna"].eq(example_name),
        ["columna", "rol_pdf", "familia_pdf", "ventana_pdf", "descripcion_breve"]
    ])


,Elemento,Definición
0,Documento,"Dataset Master NBA - Especificación funcional,..."
1,Versión,1.0
2,Fecha,2026-06-22
3,Estado,Diseño previo a implementación
4,Unidad de observación,Jugador en un partido objetivo
5,Llave lógica,PLAYER_ID + TARGET_GAME_ID
6,Campos jugador,419
7,Campos rival incorporados,321
8,Campos master esperados,740
9,Regla temporal,Toda feature usa información anterior a TARGET...


### Diccionario interpretado

Metadata funcional aplicada a las columnas del dataset

,Métrica,Valor
0,Columnas interpretadas,419
1,Predictores permitidos,385
2,Campos excluidos,34


,columna,rol_pdf,familia_pdf,metrica_nba,estadistico,ventana_pdf,descripcion_breve,predictor_permitido,alerta
0,PLAYER_ID,Clave,Identidad,variable de contexto,valor,No aplica,Identificador NBA estable del jugador.,False,Identificador; conservar solo para trazabilidad.
1,PLAYER_NAME,Contexto,Identidad,variable de contexto,valor,No aplica,Nombre canónico del jugador.,True,
2,PLAYER_NAME_RAW,Contexto,Identidad,variable de contexto,valor,No aplica,Nombre original conservado para auditoría.,True,
3,TARGET_GAME_ID,Clave,Partido objetivo,variable de contexto,valor,No aplica,Identificador del partido objetivo.,False,Resultado o contexto del partido objetivo; rev...
4,TARGET_GAME_DATE,Clave temporal,Partido objetivo,variable de contexto,valor,No aplica,Fecha del partido objetivo.,False,Resultado o contexto del partido objetivo; rev...
5,TARGET_SEASON_YEAR,Contexto,Partido objetivo,variable de contexto,valor,SEASON,Temporada NBA del partido objetivo.,False,Resultado o contexto del partido objetivo; rev...
6,TARGET_SEASON_TYPE,Contexto,Partido objetivo,variable de contexto,valor,SEASON,Tipo de temporada del partido objetivo.,False,Resultado o contexto del partido objetivo; rev...
7,TARGET_TEAM_ID,Contexto,Partido objetivo,variable de contexto,valor,No aplica,Variable de contexto del jugador.,False,Resultado o contexto del partido objetivo; rev...
8,TARGET_TEAM_ABBREVIATION,Contexto,Partido objetivo,variable de contexto,valor,No aplica,Variable de contexto del jugador.,False,Resultado o contexto del partido objetivo; rev...
9,TARGET_TEAM_NAME,Contexto,Partido objetivo,variable de contexto,valor,No aplica,Variable de contexto del jugador.,False,Resultado o contexto del partido objetivo; rev...


,columna,rol_pdf,familia_pdf,ventana_pdf,descripcion_breve
172,MIN_DREB_L10,Predictor,Agregado jugador L10,L10,Mínimo de rebotes defensivos del jugador en lo...


## 6. Estandarización Min-Max `[0,1]`

Para cada variable numérica no identificadora:

\[
x'=\frac{x-\min(x)}{\max(x)-\min(x)}
\]

- `df` conserva los valores originales.
- `df_scaled` contiene las variables numéricas transformadas a `[0,1]`.
- `df_eda` conserva `TARGET_FG3M` en escala original.
- Las variables constantes quedan en `0.0` y se reportan.


In [8]:
# @title
def is_identifier_like(col):
    return (
        col in CONFIG["ID_COLS"]
        or col.endswith("_ID")
        or col.endswith("GAME_ID")
    )

df_scaled = df.copy()
scaling_rows = []

numeric_columns = df.select_dtypes(include=np.number).columns.tolist()

for col in numeric_columns:
    series = pd.to_numeric(df[col], errors="coerce")
    original_min = series.min()
    original_max = series.max()

    if is_identifier_like(col):
        status = "excluida_identificador"
    elif series.notna().sum() == 0:
        status = "sin_datos"
        df_scaled[col] = np.nan
    elif original_max == original_min:
        status = "constante"
        df_scaled[col] = np.where(series.notna(), CONFIG["SCALING_MIN"], np.nan)
    else:
        status = "normalizada"
        low = CONFIG["SCALING_MIN"]
        high = CONFIG["SCALING_MAX"]
        df_scaled[col] = low + (series - original_min) * (high - low) / (original_max - original_min)

    scaling_rows.append({
        "columna": col,
        "mínimo_original": original_min,
        "máximo_original": original_max,
        "mínimo_escalado": pd.to_numeric(df_scaled[col], errors="coerce").min(),
        "máximo_escalado": pd.to_numeric(df_scaled[col], errors="coerce").max(),
        "estado": status,
    })

scaling_report = pd.DataFrame(scaling_rows)
df_eda = df_scaled.copy()

if CONFIG["TARGET_COL"] in df.columns:
    df_eda[CONFIG["TARGET_COL"]] = pd.to_numeric(df[CONFIG["TARGET_COL"]], errors="coerce")

section("Resultado de la estandarización")
metric_table({
    "Numéricas": f"{len(numeric_columns):,}",
    "Normalizadas": f"{(scaling_report['estado'] == 'normalizada').sum():,}",
    "Constantes": f"{(scaling_report['estado'] == 'constante').sum():,}",
    "Identificadores excluidos": f"{(scaling_report['estado'] == 'excluida_identificador').sum():,}",
})

show_df(scaling_report.sort_values(["estado", "columna"]), 30)


### Resultado de la estandarización

,Métrica,Valor
0,Numéricas,398
1,Normalizadas,394
2,Constantes,0
3,Identificadores excluidos,4


,columna,mínimo_original,máximo_original,mínimo_escalado,máximo_escalado,estado
8,LAST_GAME_ID,"22,500,712.000","22,500,867.000","22,500,712.000","22,500,867.000",excluida_identificador
0,PLAYER_ID,"2,544.000","1,643,133.000","2,544.000","1,643,133.000",excluida_identificador
1,TARGET_GAME_ID,"22,500,717.000","22,500,878.000","22,500,717.000","22,500,878.000",excluida_identificador
2,TARGET_TEAM_ID,"1,610,612,737.000","1,610,612,766.000","1,610,612,737.000","1,610,612,766.000",excluida_identificador
3,AGE,19.000,41.000,0.000,1.000,normalizada
313,AST_PER_MIN,0.000,0.501,0.000,1.000,normalizada
329,AST_PER_MIN_L10,0.000,0.688,0.000,1.000,normalizada
345,AST_PER_MIN_L20,0.000,0.984,0.000,1.000,normalizada
68,AST_TOTAL,0.000,508.000,0.000,1.000,normalizada
158,AST_TOTAL_L10,0.000,106.000,0.000,1.000,normalizada


## 7. Estructura y familias de variables


In [19]:
# @title
structure_summary = (
    variable_dictionary
    .groupby(["rol_pdf", "familia_pdf", "ventana_pdf"], dropna=False)
    .agg(
        variables=("columna", "count"),
        predictores_permitidos=("predictor_permitido", "sum")
    )
    .reset_index()
    .sort_values("variables", ascending=False)
)

display(structure_summary.head(40))

family_counts = variable_dictionary["familia_pdf"].value_counts().sort_values()

fig = px.bar(
    x=family_counts.values,
    y=family_counts.index,
    orientation='h',
    title="Cantidad de variables por familia funcional",
    labels={'x': "Número de variables", 'y': "Familia funcional"},
    color_discrete_sequence=[BLUE],
    height=max(500, len(family_counts) * 35)
)

fig.update_traces(texttemplate='%{x}', textposition='outside')
fig.update_layout(uniformtext_minsize=8, uniformtext_mode='hide')
fig = finish_plotly_layout(fig, title="Cantidad de variables por familia funcional",
                           xlabel="Número de variables", ylabel="Familia funcional")
fig.show()

,rol_pdf,familia_pdf,ventana_pdf,variables,predictores_permitidos
23,Predictor/contexto,Agregado jugador SEASON,SEASON,96,96
13,Predictor,Agregado jugador L20,L20,95,95
12,Predictor,Agregado jugador L10,L10,95,95
15,Predictor,Tasa jugador,L20,16,16
14,Predictor,Tasa jugador,L10,16,16
16,Predictor,Tasa jugador,SEASON,16,16
21,Predictor,Último partido,Último partido,12,12
25,Target,Resultado target,SEASON,11,0
1,Alias target,Compatibilidad,SEASON,11,0
22,Predictor/contexto,Agregado jugador SEASON,No aplica,9,9


## 8. Calidad de datos y controles temporales


In [9]:
# @title
def quality_report(data):
    rows = []

    for col in data.columns:
        series = data[col]
        numeric = pd.api.types.is_numeric_dtype(series)
        null_count = int(series.isna().sum())
        unique_count = int(series.nunique(dropna=True))

        row = {
            "columna": col,
            "tipo": str(series.dtype),
            "nulos": null_count,
            "nulos_pct": null_count / max(len(data), 1),
            "únicos": unique_count,
            "constante": unique_count <= 1,
            "ceros_pct": (
                pd.to_numeric(series, errors="coerce").eq(0).mean()
                if numeric else np.nan
            ),
            "negativos": (
                int(pd.to_numeric(series, errors="coerce").lt(0).sum())
                if numeric else np.nan
            ),
            "infinitos": (
                int(np.isinf(pd.to_numeric(series, errors="coerce")).sum())
                if numeric else np.nan
            ),
        }

        warnings_list = []
        if row["nulos_pct"] >= CONFIG["NULL_THRESHOLD"]:
            warnings_list.append("nulos_altos")
        if row["constante"]:
            warnings_list.append("constante")
        if numeric and row["infinitos"] > 0:
            warnings_list.append("infinitos")

        meta = variable_dictionary.loc[variable_dictionary["columna"].eq(col)]
        if not meta.empty and not bool(meta.iloc[0]["predictor_permitido"]):
            warnings_list.append("excluir_de_X")

        row["estado"] = "ok" if not warnings_list else " | ".join(warnings_list)
        rows.append(row)

    return pd.DataFrame(rows)

quality_df = quality_report(df)

section("Resumen de calidad")
metric_table({
    "Columnas con nulos >= umbral": f"{(quality_df['nulos_pct'] >= CONFIG['NULL_THRESHOLD']).sum():,}",
    "Columnas constantes": f"{quality_df['constante'].sum():,}",
    "Columnas con infinitos": f"{(quality_df['infinitos'].fillna(0) > 0).sum():,}",
    "Columnas marcadas para excluir de X": f"{quality_df['estado'].str.contains('excluir_de_X').sum():,}",
})

display(
    quality_df.loc[quality_df["estado"].ne("ok")]
    .sort_values(["nulos_pct", "columna"], ascending=[False, True])
    .head(60)
)

if {"PLAYER_ID", "TARGET_GAME_ID"}.issubset(df.columns):
    duplicate_keys = int(df.duplicated(["PLAYER_ID", "TARGET_GAME_ID"]).sum())
    note(f"Duplicados en `PLAYER_ID + TARGET_GAME_ID`: **{duplicate_keys:,}**.", "Control de llave")

if {"LAST_GAME_DATE", "TARGET_GAME_DATE"}.issubset(df.columns):
    last_date = pd.to_datetime(df["LAST_GAME_DATE"], errors="coerce")
    target_date = pd.to_datetime(df["TARGET_GAME_DATE"], errors="coerce")
    invalid_temporal = int((last_date >= target_date).fillna(False).sum())
    note(
        f"Filas donde `LAST_GAME_DATE >= TARGET_GAME_DATE`: **{invalid_temporal:,}**. "
        "El valor esperado es cero.",
        "Control temporal"
    )


### Resumen de calidad

,Métrica,Valor
0,Columnas con nulos >= umbral,32
1,Columnas constantes,2
2,Columnas con infinitos,0
3,Columnas marcadas para excluir de X,34


,columna,tipo,nulos,nulos_pct,únicos,constante,ceros_pct,negativos,infinitos,estado
350,AST_PER_MIN_L10,float64,2164,0.693,687,False,0.003,0.000,0.000,nulos_altos
353,BLK_PER_MIN_L10,float64,2164,0.693,504,False,0.048,0.000,0.000,nulos_altos
348,DREB_PER_MIN_L10,float64,2164,0.693,714,False,0.002,0.000,0.000,nulos_altos
344,FG3A_PER_MIN_L10,float64,2164,0.693,691,False,0.019,0.000,0.000,nulos_altos
343,FG3M_PER_MIN_L10,float64,2164,0.693,582,False,0.030,0.000,0.000,nulos_altos
342,FGA_PER_MIN_L10,float64,2164,0.693,782,False,0.000,0.000,0.000,nulos_altos
341,FGM_PER_MIN_L10,float64,2164,0.693,717,False,0.001,0.000,0.000,nulos_altos
346,FTA_PER_MIN_L10,float64,2164,0.693,642,False,0.016,0.000,0.000,nulos_altos
345,FTM_PER_MIN_L10,float64,2164,0.693,628,False,0.020,0.000,0.000,nulos_altos
347,OREB_PER_MIN_L10,float64,2164,0.693,641,False,0.009,0.000,0.000,nulos_altos


> **Control de llave:** Duplicados en `PLAYER_ID + TARGET_GAME_ID`: **0**.

> **Control temporal:** Filas donde `LAST_GAME_DATE >= TARGET_GAME_DATE`: **0**. El valor esperado es cero.

## 9. Distribución de `TARGET_FG3M`

`TARGET_FG3M` es una variable discreta de conteo. Se inspeccionan frecuencia, dispersión y valores extremos.


In [21]:
# @title
target_col = CONFIG["TARGET_COL"]

if target_col not in df.columns:
    raise KeyError(f"No existe la variable dependiente {target_col}.")

target_series = pd.to_numeric(df[target_col], errors="coerce").dropna()

target_stats = pd.DataFrame({
    "estadístico": [
        "observaciones", "media", "mediana", "varianza", "desviación estándar",
        "mínimo", "máximo", "ratio varianza/media", "asimetría"
    ],
    "valor": [
        len(target_series),
        target_series.mean(),
        target_series.median(),
        target_series.var(),
        target_series.std(),
        target_series.min(),
        target_series.max(),
        target_series.var() / max(target_series.mean(), 1e-12),
        target_series.skew(),
    ]
})
display(target_stats)

frequency = target_series.value_counts().sort_index().reset_index()
frequency.columns = [target_col, 'frecuencia']

fig_hist = px.bar(
    frequency,
    x=target_col,
    y='frecuencia',
    title=f"Distribución observada de {target_col}",
    labels={'frecuencia': "Frecuencia", target_col: "Triples convertidos"},
    color_discrete_sequence=[BLUE],
    text_auto=True
)
fig_hist.update_traces(textposition='outside')
fig_hist = finish_plotly_layout(fig_hist, xlabel="Triples convertidos", ylabel="Frecuencia")
fig_hist.show()

fig_box = go.Figure()
fig_box.add_trace(go.Box(
    y=target_series.values,
    name="Distribución",
    marker_color=LIGHT_BLUE,
    boxmean=True,
    boxpoints='outliers',
    jitter=0.3,
    pointpos=-1.8,
    hoverlabel=dict(namelength=-1)
))
fig_box.update_layout(
    title_text=f"Dispersión y valores extremos de {target_col}",
    showlegend=False
)
fig_box = finish_plotly_layout(fig_box, xlabel="", ylabel="Triples convertidos")
fig_box.show()

,estadístico,valor
0,observaciones,"3,122.000"
1,media,1.249
2,mediana,1.000
3,varianza,2.211
4,desviación estándar,1.487
5,mínimo,0.000
6,máximo,12.000
7,ratio varianza/media,1.770
8,asimetría,1.477


## 10. Asociación lineal con la variable dependiente

Se calculan correlaciones de Pearson entre `TARGET_FG3M` y las variables numéricas permitidas como predictores. La magnitud absoluta indica fuerza de asociación lineal; el signo indica dirección.


In [20]:
# @title
allowed_predictors = set(
    variable_dictionary.loc[
        variable_dictionary["predictor_permitido"],
        "columna"
    ]
)

numeric_predictors = [
    col for col in df.select_dtypes(include=np.number).columns
    if col in allowed_predictors
    and col != target_col
    and not is_identifier_like(col)
    and pd.to_numeric(df[col], errors="coerce").nunique(dropna=True) > 1
]

corrs_signed = (
    df[numeric_predictors]
    .apply(pd.to_numeric, errors="coerce")
    .corrwith(pd.to_numeric(df[target_col], errors="coerce"))
    .dropna()
    .sort_values(key=lambda s: s.abs(), ascending=False)
)

top_n = min(30, len(corrs_signed))
top_corr = corrs_signed.head(top_n)

corr_table = pd.DataFrame({
    "variable": top_corr.index,
    "pearson_r": top_corr.values,
    "abs_r": top_corr.abs().values,
}).merge(
    variable_dictionary[
        ["columna", "familia_pdf", "ventana_pdf", "descripcion_breve"]
    ],
    left_on="variable",
    right_on="columna",
    how="left",
).drop(columns="columna")

display(corr_table.head(30))

ordered_corr = top_corr.sort_values(ascending=True)

fig = px.bar(
    x=ordered_corr.values,
    y=ordered_corr.index,
    orientation='h',
    title=f"Principales asociaciones con {target_col}",
    labels={'x': "Correlación de Pearson", 'y': ""},
    color=ordered_corr.apply(lambda x: "Positiva" if x >= 0 else "Negativa"),
    color_discrete_map={'Positiva': BLUE, 'Negativa': RED},
    height=max(500, top_n * 20),
    text_auto=".3f",
)

fig.update_traces(textposition='outside')
fig.update_layout(
    title_text=f"Principales asociaciones con {target_col}",
    xaxis_title="Correlación de Pearson",
    showlegend=True,
    legend_title_text="Dirección",
    xaxis_range=[-1, 1] # Ensure consistent range for correlations
)

fig.add_vline(x=0.30, line_dash="dash", line_color=GOLD, annotation_text="|r| = 0.30", annotation_position="top right")
fig.add_vline(x=0.50, line_dash="dot", line_color=NAVY, annotation_text="|r| = 0.50", annotation_position="top right")
fig = finish_plotly_layout(fig, xlabel="|Correlación de Pearson|", ylabel="")
fig.show()

note(
    "Azul indica correlación positiva y rojo correlación negativa. "
    "La correlación no implica causalidad y puede reflejar variables redundantes.",
    "Interpretación"
)

,variable,pearson_r,abs_r,familia_pdf,ventana_pdf,descripcion_breve
0,AVG_FG3A_L20,0.539,0.539,Agregado jugador L20,L20,Promedio de triples intentados del jugador en ...
1,AVG_FG3M,0.538,0.538,Agregado jugador SEASON,SEASON,Promedio de triples convertidos del jugador en...
2,AVG_FG3A_L10,0.537,0.537,Agregado jugador L10,L10,Promedio de triples intentados del jugador en ...
3,FG3A_TOTAL_L20,0.537,0.537,Agregado jugador L20,L20,Suma de triples intentados del jugador en los ...
4,AVG_FG3A,0.536,0.536,Agregado jugador SEASON,SEASON,Promedio de triples intentados del jugador en ...
5,FG3A_TOTAL_L10,0.536,0.536,Agregado jugador L10,L10,Suma de triples intentados del jugador en los ...
6,AVG_FG3M_L20,0.534,0.534,Agregado jugador L20,L20,Promedio de triples convertidos del jugador en...
7,FG3M_TOTAL_L20,0.533,0.533,Agregado jugador L20,L20,Suma de triples convertidos del jugador en los...
8,AVG_FG3M_L10,0.521,0.521,Agregado jugador L10,L10,Promedio de triples convertidos del jugador en...
9,FG3M_TOTAL_L10,0.520,0.520,Agregado jugador L10,L10,Suma de triples convertidos del jugador en los...


> **Interpretación:** Azul indica correlación positiva y rojo correlación negativa. La correlación no implica causalidad y puede reflejar variables redundantes.

## 11. Dispersión con línea de tendencia

GitHub no ejecuta widgets interactivos. Para mantener el notebook completamente renderizable, la variable se selecciona mediante `CONFIG["SCATTER_VARIABLE"]`.

Cambia ese valor en la configuración y vuelve a ejecutar esta celda. También se genera una galería estática con las variables más asociadas al target.


In [23]:
# @title
def scatter_with_trend(x_col, show=True):
    if x_col not in df_eda.columns:
        raise KeyError(f"La variable {x_col} no existe en el dataset.")

    plot_df = pd.DataFrame({
        "x": pd.to_numeric(df_eda[x_col], errors="coerce"),
        "y": pd.to_numeric(df[target_col], errors="coerce"),
    }).dropna()

    if len(plot_df) < 3:
        raise ValueError(f"No hay suficientes observaciones válidas para {x_col}.")

    corr_p = plot_df["x"].corr(plot_df["y"], method="pearson")
    corr_s = plot_df["x"].corr(plot_df["y"], method="spearman")

    if len(plot_df) > CONFIG["SCATTER_MAX_POINTS"]:
        sample = plot_df.sample(CONFIG["SCATTER_MAX_POINTS"], random_state=CONFIG["SEED"])
    else:
        sample = plot_df

    fig = px.scatter(
        sample,
        x="x",
        y="y",
        title=f"{target_col} vs {x_col}",
        labels={
            "x": f"{x_col} - escala Min-Max [0,1]",
            "y": f"{target_col} - triples convertidos",
        },
        color_discrete_sequence=[BLUE],
        opacity=0.28
    )

    slope = np.nan
    intercept = np.nan

    if plot_df["x"].nunique() > 1:
        slope, intercept = np.polyfit(plot_df["x"], plot_df["y"], 1)
        x_line = np.linspace(plot_df["x"].min(), plot_df["x"].max(), 200)
        fig.add_trace(go.Scatter(
            x=x_line,
            y=intercept + slope * x_line,
            mode='lines',
            name=f"Tendencia lineal: y = {intercept:.2f} + {slope:.2f}x",
            line=dict(color=RED, width=2.4),
        ))

        unique_bins = min(10, plot_df["x"].nunique())
        if unique_bins >= 3:
            bins = pd.qcut(plot_df["x"], q=unique_bins, duplicates="drop")
            binned = plot_df.groupby(bins, observed=True, as_index=False).agg(x=("x", "mean"), y=("y", "mean"))
            fig.add_trace(go.Scatter(
                x=binned["x"],
                y=binned["y"],
                mode='lines+markers',
                name="Promedio por cuantiles",
                marker=dict(symbol='circle', size=5, color=GOLD),
                line=dict(color=GOLD, width=1.6),
            ))

    fig = finish_plotly_layout(
        fig,
        title=f"{target_col} vs {x_col}",
        xlabel=f"{x_col} - escala Min-Max [0,1]",
        ylabel=f"{target_col} - triples convertidos",
    )

    if show:
        fig.show()

    return {
        "variable": x_col,
        "n": len(plot_df),
        "pearson_r": corr_p,
        "spearman_rho": corr_s,
        "pendiente": slope,
        "intercepto": intercept,
        "figura": fig, # Return the Plotly figure
    }

selected_scatter = CONFIG["SCATTER_VARIABLE"]

if selected_scatter not in numeric_predictors:
    selected_scatter = corrs_signed.index[0]
    note(
        f"La variable configurada no estaba disponible. Se seleccionó automáticamente `{selected_scatter}`.",
        "Selección automática"
    )

scatter_result = scatter_with_trend(selected_scatter)

display(pd.DataFrame([{
    "variable": scatter_result["variable"],
    "n": scatter_result["n"],
    "pearson_r": scatter_result["pearson_r"],
    "spearman_rho": scatter_result["spearman_rho"],
    "pendiente": scatter_result["pendiente"],
}]))

description_lookup = variable_dictionary.set_index("columna")["descripcion_breve"].to_dict()
note(description_lookup.get(selected_scatter, "Variable explicativa."), "Descripción")

gallery_variables = corrs_signed.head(CONFIG["SCATTER_GALLERY_TOP"]).index.tolist()

for variable in gallery_variables:
    if variable != selected_scatter:
        scatter_with_trend(variable)

,variable,n,pearson_r,spearman_rho,pendiente
0,AVG_FG3A_L20,3122,0.539,0.562,3.914


> **Descripción:** Promedio de triples intentados del jugador en los últimos 20 partidos previos.

## 12. Clustering jerárquico de variables

El clustering agrupa **variables**, no jugadores. Cada variable se representa por su vector de valores normalizados a través de los partidos.

- Con `ward`, se utiliza distancia euclidiana sobre variables Min-Max.
- Para otros métodos se utiliza distancia `1 - |r|`.
- El target, identificadores, aliases `NEXT_GAME_*` y otros targets se excluyen.


In [24]:
# @title
import plotly.figure_factory as ff

cluster_candidates = [
    col for col in numeric_predictors
    if col in df_scaled.columns
    and pd.to_numeric(df_scaled[col], errors="coerce").nunique(dropna=True) > 1
]

cluster_data = df_scaled[cluster_candidates].apply(pd.to_numeric, errors="coerce")

if len(cluster_data) > CONFIG["CLUSTER_MAX_ROWS"]:
    cluster_data = cluster_data.sample(
        CONFIG["CLUSTER_MAX_ROWS"],
        random_state=CONFIG["SEED"]
    )

coverage = cluster_data.notna().mean()
cluster_data = cluster_data.loc[:, coverage >= 0.50]
cluster_imputed = cluster_data.fillna(cluster_data.median())

if cluster_imputed.shape[1] < 2:
    raise ValueError("Se requieren al menos dos variables explicativas válidas para clustering.")

linkage_method = CONFIG["CLUSTER_LINKAGE"].lower()

if linkage_method == "ward":
    Z = linkage(cluster_imputed.T.values, method="ward", metric="euclidean")
    distance_label = "Distancia euclidiana acumulada"
    cluster_basis = "Ward sobre variables Min-Max"
else:
    corr_matrix = cluster_imputed.corr().abs()
    distance_matrix = 1 - corr_matrix
    np.fill_diagonal(distance_matrix.values, 0)
    condensed = squareform(np.maximum(distance_matrix.values, 0), checks=False)
    Z = linkage(condensed, method=linkage_method)
    distance_label = "Distancia 1 - |r|"
    cluster_basis = f"{linkage_method.title()} sobre correlación absoluta"

labels = fcluster(
    Z,
    t=CONFIG["CLUSTER_N_GROUPS"],
    criterion="maxclust"
)

cluster_map = pd.DataFrame({
    "columna": cluster_imputed.columns,
    "cluster": labels,
})
cluster_map["cluster_label"] = cluster_map["cluster"].map(lambda value: f"G{value:02d}")

cluster_map = cluster_map.merge(
    variable_dictionary[
        [
            "columna", "rol_pdf", "familia_pdf", "metrica_nba",
            "estadistico", "ventana_pdf", "descripcion_breve"
        ]
    ],
    on="columna",
    how="left",
)

# Plotly Dendrogram
fig_dendro = ff.create_dendrogram(
    cluster_imputed.T.values,
    orientation='right',
    labels=cluster_imputed.columns.tolist(),
    linkagefun=lambda x: Z
)
fig_dendro.update_layout(
    title_text=f"Dendrograma de variables - {linkage_method.title()} - {CONFIG['CLUSTER_N_GROUPS']} grupos",
    height=max(600, cluster_imputed.shape[1] * 12),
    xaxis_title=distance_label,
    yaxis_title="Variables ordenadas jerárquicamente"
)
fig_dendro = finish_plotly_layout(fig_dendro)
fig_dendro.show()

cluster_sizes = cluster_map["cluster_label"].value_counts().sort_index()

# Plotly Bar chart for cluster sizes
fig_sizes = px.bar(
    x=cluster_sizes.index,
    y=cluster_sizes.values,
    title="Tamaño de los grupos de variables",
    labels={'x': "Cluster", 'y': "Número de variables"},
    color=cluster_sizes.index,
    color_discrete_sequence=PALETTE,
    text_auto=True
)
fig_sizes.update_traces(textposition='outside')
fig_sizes = finish_plotly_layout(fig_sizes, xlabel="Cluster", ylabel="Número de variables")
fig_sizes.show()

note(
    f"Base del clustering: **{cluster_basis}**. "
    f"Se utilizaron {cluster_imputed.shape[1]:,} variables y {cluster_imputed.shape[0]:,} observaciones.",
    "Metodología"
)

> **Metodología:** Base del clustering: **Ward sobre variables Min-Max**. Se utilizaron 340 variables y 3,122 observaciones.

## 13. Perfil e interpretación automática de los clusters

La interpretación combina:

1. número de variables;
2. familia funcional dominante según la especificación;
3. ventana temporal dominante;
4. métrica NBA dominante;
5. variable con mayor asociación absoluta con `TARGET_FG3M`.

Los nombres son descripciones analíticas, no clases oficiales del dataset.


In [26]:
# @title
def dominant_value(series):
    valid = series.dropna()
    if valid.empty:
        return "Sin clasificar"
    return valid.value_counts().index[0]

def semantic_cluster_label(metrics, families):
    text = " ".join(metrics.astype(str).tolist() + families.astype(str).tolist()).upper()

    if "FG3" in text or "TRIPLES" in text:
        return "Volumen y producción de triples"
    if any(token in text for token in ["FGA", "FGM", "MINUTOS"]):
        return "Minutos y volumen general de tiro"
    if any(token in text for token in ["REBOTES", "OREB", "DREB"]):
        return "Perfil interior y rebotes"
    if any(token in text for token in ["ASISTENCIAS", "PÉRDIDAS", "PUNTOS"]):
        return "Uso, creación y actividad ofensiva"
    if "RENDIMIENTO CONTRA EL RIVAL" in text:
        return "Contexto permitido por el rival"
    if "PRODUCCIÓN RIVAL" in text:
        return "Producción y forma del rival"
    if "COBERTURA" in text:
        return "Cobertura y disponibilidad histórica"
    return "Perfil estadístico mixto"

profile_rows = []

for cluster_id in sorted(cluster_map["cluster"].unique()):
    subset = cluster_map.loc[cluster_map["cluster"].eq(cluster_id)].copy()
    columns_cluster = subset["columna"].tolist()
    cluster_corr = corrs_signed.reindex(columns_cluster).dropna()

    if cluster_corr.empty:
        representative = columns_cluster[0]
        representative_r = np.nan
    else:
        representative = cluster_corr.abs().idxmax()
        representative_r = cluster_corr.loc[representative]

    label = semantic_cluster_label(
        subset["metrica_nba"],
        subset["familia_pdf"],
    )

    profile_rows.append({
        "cluster": f"G{cluster_id:02d}",
        "variables": len(subset),
        "nombre_funcional": label,
        "familia_dominante": dominant_value(subset["familia_pdf"]),
        "ventana_dominante": dominant_value(subset["ventana_pdf"]),
        "metrica_dominante": dominant_value(subset["metrica_nba"]),
        "representante": representative,
        "pearson_r_representante": representative_r,
        "interpretacion": (
            f"El grupo concentra principalmente variables de {label.lower()}. "
            f"Su variable más asociada con {target_col} es {representative} "
            f"(r={representative_r:+.3f})."
            if pd.notna(representative_r)
            else f"El grupo concentra principalmente variables de {label.lower()}."
        ),
    })

cluster_profile = pd.DataFrame(profile_rows)
display(cluster_profile)

plot_profile = cluster_profile.copy()
plot_profile["abs_r"] = plot_profile["pearson_r_representante"].abs()

fig_bar = px.bar(
    plot_profile,
    x="abs_r",
    y="cluster",
    orientation='h',
    title=f"Mayor asociación con {target_col} dentro de cada cluster",
    labels={'abs_r': "|Correlación de Pearson|", 'cluster': "Cluster"},
    color="cluster",
    color_discrete_sequence=PALETTE,
    text=plot_profile.apply(lambda row: f"{row['representante']} ({row['pearson_r_representante']:+.3f})" if pd.notna(row['pearson_r_representante']) else row['representante'], axis=1)
)
fig_bar.update_traces(textposition='outside')
fig_bar = finish_plotly_layout(fig_bar, xlabel="|Correlación de Pearson|", ylabel="Cluster")
fig_bar.show()

representatives = cluster_profile["representante"].dropna().tolist()
representative_data = df[representatives + [target_col]].apply(pd.to_numeric, errors="coerce")
representative_corr = representative_data.corr()

fig_heatmap = px.imshow(
    representative_corr,
    text_auto=".2f",
    aspect="auto",
    color_continuous_scale=px.colors.sequential.Bluered,
    range_color=[-1, 1],
    title="Correlación entre representantes de clusters",
    labels=dict(color="Pearson r")
)
fig_heatmap.update_layout(
    xaxis_tickangle=-45
)
fig_heatmap = finish_plotly_layout(fig_heatmap, xlabel="", ylabel="")
fig_heatmap.show()


,cluster,variables,nombre_funcional,familia_dominante,ventana_dominante,metrica_dominante,representante,pearson_r_representante,interpretacion
0,G01,78,Volumen y producción de triples,Agregado jugador L20,SEASON,minutos jugados,MIN_FG3A_L10,0.455,El grupo concentra principalmente variables de...
1,G02,19,Volumen y producción de triples,Tendencia jugador,L10,variable de contexto,MAX_MIN,0.269,El grupo concentra principalmente variables de...
2,G03,32,Perfil interior y rebotes,Agregado jugador L10,SEASON,rebotes ofensivos,REB_PER_MIN,-0.214,El grupo concentra principalmente variables de...
3,G04,88,"Uso, creación y actividad ofensiva",Agregado jugador SEASON,SEASON,tiros libres intentados,AVG_FGA_L20,0.371,El grupo concentra principalmente variables de...
4,G05,26,Volumen y producción de triples,Agregado jugador SEASON,SEASON,triples convertidos,AVG_FG3A_L20,0.539,El grupo concentra principalmente variables de...
5,G06,97,Volumen y producción de triples,Agregado jugador SEASON,SEASON,robos,MAX_FGA_L20,0.341,El grupo concentra principalmente variables de...


## 14. Ventanas temporales relacionadas con triples

Según la especificación, las variables de jugador utilizan:

- temporada hasta antes del target;
- últimos 10 partidos;
- últimos 20 partidos;
- tendencias L3, L5 y L10.

La comparación se limita a columnas realmente presentes en el dataset.


In [27]:
# @title
window_columns = {
    "SEASON": "AVG_FG3M",
    "L10": "AVG_FG3M_L10",
    "L20": "AVG_FG3M_L20",
}

available_windows = {
    label: column
    for label, column in window_columns.items()
    if column in df.columns
}

if len(available_windows) >= 2:
    window_frame = pd.DataFrame({
        label: pd.to_numeric(df[column], errors="coerce")
        for label, column in available_windows.items()
    })

    fig_box = px.box(
        window_frame,
        title="Distribución de promedios históricos de triples por ventana",
        labels={'variable': "Ventana histórica", 'value': "Promedio de FG3M"},
        color_discrete_sequence=PALETTE,
        points="all"
    )
    fig_box.update_layout(
        title_text="Distribución de promedios históricos de triples por ventana"
    )
    fig_box = finish_plotly_layout(fig_box, xlabel="Ventana histórica", ylabel="Promedio de FG3M")
    fig_box.show()

    window_summary = window_frame.describe().T
    window_summary["cv"] = window_summary["std"] / window_summary["mean"].replace(0, np.nan)
    display(window_summary)
else:
    note(
        "No se encontraron al menos dos columnas entre `AVG_FG3M`, `AVG_FG3M_L10` y `AVG_FG3M_L20`.",
        "Ventanas"
    )

trend_columns = [
    column for column in [
        "FG3M_TREND_L3", "FG3M_TREND_L5", "FG3M_TREND_L10",
        "FG3_PCT_TREND_L3", "FG3_PCT_TREND_L5", "FG3_PCT_TREND_L10"
    ]
    if column in df.columns
]

if trend_columns:
    trend_summary = df[trend_columns].apply(pd.to_numeric, errors="coerce").describe().T
    display(trend_summary)


,count,mean,std,min,25%,50%,75%,max,cv
SEASON,"3,122.000",1.218,0.885,0.000,0.519,1.071,1.829,3.718,0.727
L10,"3,122.000",1.246,0.934,0.000,0.500,1.100,1.800,4.700,0.749
L20,"3,122.000",1.232,0.908,0.000,0.500,1.100,1.800,4.300,0.737


,count,mean,std,min,25%,50%,75%,max
FG3M_TREND_L3,"3,114.000",0.002,0.870,-3.500,-0.500,0.000,0.500,4.500
FG3M_TREND_L5,"3,114.000",0.010,0.390,-1.500,-0.200,0.000,0.200,2.500
FG3M_TREND_L10,"3,114.000",0.006,0.148,-1.500,-0.073,0.000,0.085,1.000
FG3_PCT_TREND_L3,"3,114.000",-0.435,10.395,-333.500,-0.100,0.000,0.100,0.500
FG3_PCT_TREND_L5,"3,114.000",-0.402,5.756,-133.400,-0.050,0.000,0.049,0.500
FG3_PCT_TREND_L10,"3,114.000",-0.176,2.165,-44.986,-0.019,0.000,0.019,11.929


## 15. Distribuciones representativas por cluster


In [28]:
# @title
for _, row in cluster_profile.iterrows():
    cluster_label = row["cluster"]
    representative = row["representante"]

    if representative not in df_eda.columns:
        continue

    series = pd.to_numeric(df_eda[representative], errors="coerce").dropna()
    if series.empty:
        continue

    cluster_number = int(cluster_label[1:])
    color = PALETTE[(cluster_number - 1) % len(PALETTE)]

    fig = px.histogram(
        series,
        nbins=25,
        title=f"{cluster_label} - {row['nombre_funcional']} - {representative}",
        labels={
            "value": "Escala Min-Max [0,1]",
            "count": "Frecuencia"
        },
        color_discrete_sequence=[color],
        opacity=0.82
    )

    fig.add_vline(
        x=series.mean(),
        line_dash="dash",
        line_color=NAVY,
        annotation_text=f"Media = {series.mean():.3f}",
        annotation_position="top right"
    )
    fig.add_vline(
        x=series.median(),
        line_dash="dot",
        line_color=RED,
        annotation_text=f"Mediana = {series.median():.3f}",
        annotation_position="top left"
    )

    fig = finish_plotly_layout(fig, xlabel="Escala Min-Max [0,1]", ylabel="Frecuencia")
    fig.show()


## 16. Resumen del EDA

El bloque final sintetiza resultados descriptivos. No debe interpretarse como evaluación predictiva ni como selección definitiva de variables.


In [16]:
# @title
mean_target = target_series.mean()
variance_target = target_series.var()
dispersion_ratio = variance_target / max(mean_target, 1e-12)

top_variables = corrs_signed.head(5)
top_variable_text = ", ".join(
    f"{name} (r={value:+.3f})"
    for name, value in top_variables.items()
)

summary_metrics = pd.DataFrame([
    ["Dataset", f"{df.shape[0]:,} filas x {df.shape[1]:,} columnas"],
    ["Target", target_col],
    ["Media del target", f"{mean_target:.3f}"],
    ["Varianza del target", f"{variance_target:.3f}"],
    ["Ratio varianza/media", f"{dispersion_ratio:.3f}"],
    ["Variables normalizadas", f"{(scaling_report['estado'] == 'normalizada').sum():,}"],
    ["Clusters", f"{cluster_map['cluster'].nunique():,}"],
    ["Principales asociaciones", top_variable_text],
    ["Fuente de metadata", f"{CONFIG['SPEC_TITLE']} v{CONFIG['SPEC_VERSION']}"],
], columns=["Elemento", "Resultado"])

display(summary_metrics)

summary_text = (
    f"### Hallazgos principales\n\n"
    f"1. **Target discreto:** `{target_col}` presenta media de **{mean_target:.3f}** "
    f"y varianza de **{variance_target:.3f}**.\n"
    f"2. **Volumen de triple:** las variables de intentos y conversiones históricas "
    f"suelen concentrar las mayores asociaciones con el target.\n"
    f"3. **Escalas comparables:** las variables numéricas analíticas están disponibles "
    f"en `df_scaled` dentro de `[0,1]`.\n"
    f"4. **Estructura redundante:** el clustering identifica grupos de variables con "
    f"comportamiento similar y ayuda a detectar colinealidad.\n"
    f"5. **Temporalidad:** `NEXT_GAME_*`, otros targets e identificadores están marcados "
    f"para exclusión de predictores.\n"
    f"6. **Alcance:** este notebook finaliza en el análisis exploratorio; no entrena "
    f"modelos ni exporta archivos."
)

display(Markdown(summary_text))


,Elemento,Resultado
0,Dataset,"3,122 filas x 419 columnas"
1,Target,TARGET_FG3M
2,Media del target,1.249
3,Varianza del target,2.211
4,Ratio varianza/media,1.770
5,Variables normalizadas,394
6,Clusters,6
7,Principales asociaciones,"AVG_FG3A_L20 (r=+0.539), AVG_FG3M (r=+0.538), ..."
8,Fuente de metadata,"Dataset Master NBA - Especificación funcional,..."


### Hallazgos principales

1. **Target discreto:** `TARGET_FG3M` presenta media de **1.249** y varianza de **2.211**.
2. **Volumen de triple:** las variables de intentos y conversiones históricas suelen concentrar las mayores asociaciones con el target.
3. **Escalas comparables:** las variables numéricas analíticas están disponibles en `df_scaled` dentro de `[0,1]`.
4. **Estructura redundante:** el clustering identifica grupos de variables con comportamiento similar y ayuda a detectar colinealidad.
5. **Temporalidad:** `NEXT_GAME_*`, otros targets e identificadores están marcados para exclusión de predictores.
6. **Alcance:** este notebook finaliza en el análisis exploratorio; no entrena modelos ni exporta archivos.